<a href="https://colab.research.google.com/github/vera2005/MLCV_course1/blob/main/Lesson6/Lesson6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Подготовка окружения

In [ ]:
# Устанавливаем библиотеку Ultralytics (содержит YOLOv8)
!pip install ultralytics -q

import torch
print("GPU доступен:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Модель GPU:", torch.cuda.get_device_name(0))

Загружаем сеть и кормим ее картинкой

In [ ]:
from ultralytics import YOLO
import cv2
from google.colab.patches import cv2_imshow

# Загружаем предобученную модель (nano-версия, самая быстрая)

# Отправляем картинку в сеть
results = model("https://ultralytics.com/images/zidane.jpg")

# Превращаем результат в картинку с рамками и показываем


Смотрим, что в результате внутри

In [ ]:
# Извлекаем найденные объекты (Bounding Boxes)
boxes = results[0].boxes

for box in boxes:
    cls_id = int(box.cls[0])          # Номер класса
    cls_name = model.names[cls_id]    # Имя класса ("person", "tie")
    conf = float(box.conf[0])         # Уверенность (Confidence) от 0 до 1
    xyxy = box.xyxy[0].tolist()       # Координаты рамки [x1, y1, x2, y2]

    print(f"Класс: {cls_name:10s} | Confidence: {conf:.2%} | Координаты: {[round(v) for v in xyxy]}")

--- Что именно нашла модель ---
Класс: person     | Confidence: 83.60% | Координаты: [115, 197, 1114, 712]
Класс: person     | Confidence: 81.90% | Координаты: [748, 42, 1143, 713]
Класс: tie        | Confidence: 29.10% | Координаты: [439, 437, 524, 709]


Пробуем на своей картинке

In [ ]:
from google.colab import files
from ultralytics import YOLO
from google.colab.patches import cv2_imshow

print("Выберите изображение на вашем компьютере:")
uploaded = files.upload()
image_path = list(uploaded.keys())[0]
print(f"\nЗагружен файл: {image_path}")

model = YOLO("yolov8n.pt")
results = model(image_path)
cv2_imshow(results[0].plot())

Загрузка датасета Африка вайб (Проект Африка, кто понял, тот понял))

In [ ]:
import os
import shutil

# 1. Удаляем всё старое, чтобы ничего не мешало
if os.path.exists("/content/african-wildlife"): shutil.rmtree("/content/african-wildlife")
if os.path.exists("/content/african-wildlife.zip"): os.remove("/content/african-wildlife.zip")
if os.path.exists("/content/images"): shutil.rmtree("/content/images")
if os.path.exists("/content/labels"): shutil.rmtree("/content/labels")

# 2. Скачиваем архив
!wget -q https://github.com/ultralytics/assets/releases/download/v0.0.0/african-wildlife.zip

# 3. Создаем ЧИСТУЮ папку и распаковываем строго туда (флаг -o игнорирует любые вопросы)
!mkdir -p /content/african-wildlife
!unzip -q -o african-wildlife.zip -d /content/african-wildlife/

print("Скачивание и распаковка завершены.")

In [ ]:
import os
import shutil

# Сносим все старые папки и архивы
for path in ["/content/datasets", "/content/clean_dataset", "/content/african-wildlife", "african-wildlife.zip"]:
    if os.path.exists(path):
        if os.path.isdir(path): shutil.rmtree(path)
        else: os.remove(path)

print(" Очистка завершена. Мы начинаем с чистого листа.")

**Дообучение**

In [ ]:
import os
import glob
from PIL import Image
import matplotlib.pyplot as plt
from ultralytics import YOLO

# Загружаем нашу обученную модель
best_model = YOLO("runs/detect/train/weights/best.pt")

# Ищем картинки для теста
test_images = glob.glob("/content/datasets/african-wildlife/val/images/*.jpg")
test_images += glob.glob("/content/datasets/african-wildlife/valid/images/*.jpg")

if not test_images:
    print("Ищем картинки по всей папке datasets...")
    test_images = glob.glob("/content/datasets/**/*.jpg", recursive=True)

if not test_images:
    print(" Картинки не найдены. Твари..")
else:
    print(f" Найдено {len(test_images)} картинок. Запускаем детекцию на первых 3-х...")

    #ПИСАТЬ ТУТА


    # Выводим результат
    pred_images = glob.glob("/content/my_project/predict_run/*.jpg")

    if pred_images:
        fig, axes = plt.subplots(1, len(pred_images), figsize=(15, 5))
        if len(pred_images) == 1: axes = [axes]
        for ax, img_path in zip(axes, pred_images):
            ax.imshow(Image.open(img_path))
            ax.axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print("Не удалось сохранить предсказания.")

Тест на своей картинке

In [ ]:
from google.colab import files
from ultralytics import YOLO
from google.colab.patches import cv2_imshow

uploaded = files.upload()

if uploaded:
    img_path = list(uploaded.keys())[0]

    # Загружаем нашу обученную модель
    best_model = YOLO("runs/detect/train/weights/best.pt")

    # Запускаем предсказание на загруженной картинке
